# Représentation vectorielle et segmentation informationnelle des marchés Polymarket

## 1. Représentation vectorielle des marchés

Chaque marché Polymarket est représenté comme un vecteur :

$$
x_m \in \mathbb{R}^n
$$

où chaque dimension correspond à une métrique quantitative observable,
calculée uniquement à partir des métadonnées fournies par l’API Gamma
(liquidité, horizon temporel, structure des issues, état du marché).

Cette représentation permet la segmentation et le regroupement des marchés
selon leur structure informationnelle et leur tradabilité.



## 2. Notations (via Api GAMMA prinoalement)

début idée métrique :

- $\text{liq}_m$ : liquidité du marché 
- $t_{\text{start},m}$ : date de création du marché 
- $t_{\text{end},m}$ : date de résolution du marché 
- $\mathcal{O}_m = \{1,\dots,N_m\}$ : ensemble des issues possibles (outcomes)
- $N_m$ : nombre total d’issues possibles
- $P_{i,m} \in [0,1]$ : probabilité implicite (prix) associée à l’issue $i$, si disponible
- $t$ : temps courant
- $\tau_m = t_{\text{end},m} - t$ : temps restant avant résolution
- $\varepsilon > 0$ : seuil numérique très petit (ex : $10^{-3}$), utilisé pour éviter les divisions par zéro ou les logarithmes non définis


## 3. métriques de structure informationnelle (via API gamma)

Ces métriques caractérisent **la manière dont l’information est encodée dans la structure du marché**,
indépendamment de toute dynamique temporelle des prix.



### 3.1 complexité informationnelle (structure des issues)

$$
N_m = |\mathcal{O}_m|
$$

Cette métrique mesure la complexité informationnelle a priori du marché.
Un nombre élevé d’issues implique une information plus fragmentée
et un marché structurellement plus complexe.



### 3.2 Incohérence probabiliste structurelle

Cette métrique n’est définie que pour les marchés multi-issues
dont les issues sont **mutuellement exclusives** et pour lesquels
l’API Gamma fournit une probabilité implicite par issue.

Dans ce cas, on définit :

$$
\Delta_m = \left| \sum_{i \in \mathcal{O}_m} P_{i,m} - 1 \right|
$$

- $\Delta_m \approx 0$ : cohérence probabiliste structurelle
- $\Delta_m > 0$ : inefficience statique (déséquilibre de la structure des prix)

Pour les marchés binaires YES/NO, cette métrique n’est **pas utilisée**,
car les prix ne forment pas une distribution de probabilité normalisée.


### 3.3 Proximité structurelle à la certitude

Si les $P_{i,m}$ sont disponibles :

$$
C_m = 1 - \max_{i \in \mathcal{O}_m} P_{i,m}
$$

- $C_m \approx 0$ : une issue domine fortement (marché quasi déterministe)
- $C_m$ élevé : marché encore incertain





## métrique de liquidité (via API Gamma)

### liquidité logarithmique 

$$
L_m = \log(1 + \text{liq}_m)
$$

ça réduit les effets d’échelle entre marchés très liquides et peu liquides ( allait certainement être pris)



## Métriques temporelles (Via API gamma)

###  Temps avant résolution

$$
\tau_m = t_{\text{end},m} - t
$$

ça mesure l’horizon temporel du marché.


###  Indicateur de fin de marché

$$
\mathbb{I}_{\text{fin}}(m) =
\begin{cases}
1 & \text{si } \tau_m < \tau_0 \\
0 & \text{sinon}
\end{cases}
$$

où $\tau_0$ est un seuil fixé (ex : 24 ou 48 heures).






In [14]:
import math
import requests
from datetime import datetime, timezone
from typing import Dict, List, Optional

GAMMA_EVENTS = "https://gamma-api.polymarket.com/events"

# ----------------------------
# Helpers
# ----------------------------
def parse_iso(dt: Optional[str]) -> Optional[datetime]:
    if not dt:
        return None
    try:
        if dt.endswith("Z"):
            dt = dt[:-1] + "+00:00"
        return datetime.fromisoformat(dt).astimezone(timezone.utc)
    except Exception:
        return None

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

def bucket_horizon_days(ttm_days: Optional[float]) -> str:
    if ttm_days is None:
        return "unknown"
    if ttm_days < 7:
        return "short"
    if ttm_days < 30:
        return "mid"
    return "long"

# ----------------------------
# Gamma API
# ----------------------------
def gamma_one_page_events(limit: int = 50, offset: int = 0) -> List[Dict]:
    r = requests.get(
        GAMMA_EVENTS,
        params={
            "closed": "false",
            "limit": str(limit),
            "offset": str(offset),
            "order": "id",
            "ascending": "false",
        },
        timeout=20,
    )
    r.raise_for_status()
    data = r.json()
    return data.get("data", data) if isinstance(data, dict) else data

# ----------------------------
# Core feature engineering
# ----------------------------
def compute_market_metrics(m: Dict, event: Dict, now: datetime) -> Dict:
    # ---- text
    question_text = (m.get("question") or m.get("slug") or "")
    question_lc = question_text.lower()

    # ---- time
    end_dt = parse_iso(m.get("endDateIso") or m.get("endDate"))
    start_dt = parse_iso(m.get("startDateIso") or m.get("startDate"))
    ttm_days = (end_dt - now).total_seconds() / 86400 if end_dt else None
    age_days = (now - start_dt).total_seconds() / 86400 if start_dt else None

    # ---- numeric
    L = safe_float(m.get("liquidityNum")) or 0.0
    V = safe_float(m.get("volumeNum")) or 0.0
    fee = safe_float(m.get("fee"))

    # ---- outcomes
    outcomes = m.get("outcomes")
    if isinstance(outcomes, list) and len(outcomes) > 0:
        n_outcomes = len(outcomes)
    else:
        n_outcomes = None

    # ---- binary heuristic (Gamma outcomes often missing)
    is_binary = 1 if (
        n_outcomes == 2
        or "up or down" in question_lc
        or question_lc.startswith("will ")
    ) else 0

    # ---- resolution source
    resolution_source = (m.get("resolutionSource") or "").strip()
    description = (m.get("description") or "")
    has_rs_field = 1 if resolution_source else 0
    has_rs_in_desc = 1 if ("resolution source" in description.lower()) else 0

    # ---- event structure
    n_markets_in_event = len(event.get("markets") or [])

    # ---- derived
    eps = 1e-9
    turnover = V / (L + eps)
    log_liq = math.log1p(L)
    log_vol = math.log1p(V)

    return {
        # ids
        "market_id": m.get("id"),
        "event_id": event.get("id"),
        "category": m.get("category") or event.get("category"),
        "question": question_text,
        "active": m.get("active"),
        "closed": m.get("closed"),

        # time
        "ttm_days": ttm_days,
        "horizon_bucket": bucket_horizon_days(ttm_days),
        "age_days": age_days,

        # liquidity / activity
        "liquidityNum": L,
        "volumeNum": V,
        "log_liquidity": log_liq,
        "log_volume": log_vol,
        "turnover": turnover,
        "is_illiquid": 1 if L < 5000 else 0,

        # structure
        "n_outcomes": n_outcomes,
        "is_binary": is_binary,
        "n_markets_in_event": n_markets_in_event,

        # resolution quality
        "fee": fee,
        "has_resolutionSource_field": has_rs_field,
        "has_resolutionSource_in_description": has_rs_in_desc,
    }


In [ ]:
from datetime import datetime, timezone

now = datetime.now(timezone.utc)

# 1) Collecte VARIÉE (plusieurs offsets)
events = []
for off in [0, 200, 500]:
    events.extend(gamma_one_page_events(limit=30, offset=off))

print(f"Fetched events: {len(events)}")

# 2) Calcul des métriques
rows = []
for ev in events:
    for m in (ev.get("markets") or []):
        rows.append(compute_market_metrics(m, ev, now))

print(f"Computed markets: {len(rows)}")


print("\nSAMPLE (10 markets):")
for r in rows[:10]:
    print(
        f"- {r['question']}\n"
        f"  L={round(r['liquidityNum'],1)} | "
        f"TTM={None if r['ttm_days'] is None else round(r['ttm_days'],2)} | "
        f"bucket={r['horizon_bucket']} | "
        f"bundle={r['n_markets_in_event']} | "
        f"binary={r['is_binary']} | "
        f"hasRS={r['has_resolutionSource_field']}"
    )

# tri léger
rows_sorted = sorted(
    rows,
    key=lambda r: (r["horizon_bucket"], -r["liquidityNum"])
)

print("\nTOP by liquidity (per horizon):")
for r in rows_sorted[:15]:
    print(
        f"- {r['question']} | "
        f"L={round(r['liquidityNum'],1)} | "
        f"TTM={None if r['ttm_days'] is None else round(r['ttm_days'],2)} | "
        f"binary={r['is_binary']} | "
        f"bundle={r['n_markets_in_event']}"
    )


Fetched events: 90
Computed markets: 106

SAMPLE (10 markets):
- Solana Up or Down - December 29, 4:50PM-4:55PM ET
  L=0.0 | TTM=0.04 | bucket=short | bundle=1 | binary=1 | hasRS=1
- Bitcoin Up or Down - December 29, 4:50PM-4:55PM ET
  L=0.0 | TTM=0.04 | bucket=short | bundle=1 | binary=1 | hasRS=1
- Ethereum Up or Down - December 29, 4:50PM-4:55PM ET
  L=0.0 | TTM=0.04 | bucket=short | bundle=1 | binary=1 | hasRS=1
- XRP Up or Down - December 29, 4:50PM-4:55PM ET
  L=0.0 | TTM=0.04 | bucket=short | bundle=1 | binary=1 | hasRS=1
- Bitcoin Up or Down - December 29, 4:45PM-4:50PM ET
  L=0.0 | TTM=0.04 | bucket=short | bundle=1 | binary=1 | hasRS=1
- XRP Up or Down - December 29, 4:45PM-4:50PM ET
  L=0.0 | TTM=0.04 | bucket=short | bundle=1 | binary=1 | hasRS=1
- Ethereum Up or Down - December 29, 4:45PM-4:50PM ET
  L=0.0 | TTM=0.04 | bucket=short | bundle=1 | binary=1 | hasRS=1
- Solana Up or Down - December 29, 4:45PM-4:50PM ET
  L=0.0 | TTM=0.04 | bucket=short | bundle=1 | binary=1 | h